# Fast Solar Orbiter merged MAG availability scan + daily diagnostics

This notebook revises the previous approach by adding a **fast pre-screen step using the Solar Orbiter Archive (SOAR)**:

1. Query SOAR metadata day-by-day for merged MAG product availability (**no full data processing**).
2. Run the full MHDTurbPy pipeline only for days that appear available.
3. Downsample diagnostics to **1-minute cadence** and summarize `beta` and `sigma_c`.
4. Save the usual MHDTurbPy files for successful days.

It keeps the requirement `SOLO_use_merged_MAG = True`.


In [ ]:
from pathlib import Path
import importlib.util

import numpy as np
import pandas as pd


def _find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "functions").is_dir() and (candidate / "pyspedas").is_dir():
            return candidate
    raise RuntimeError("Could not locate MHDTurbPy root (missing functions/ and pyspedas/).")


root_dir = _find_repo_root(Path.cwd())
path_setup_file = root_dir / "functions" / "path_setup.py"
spec = importlib.util.spec_from_file_location("mhdturbpy_path_setup", path_setup_file)
if spec is None or spec.loader is None:
    raise RuntimeError(f"Could not load path setup from {path_setup_file}")
path_setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(path_setup)

root_dir = path_setup.ensure_project_paths(
    start=Path.cwd(),
    include_downloading_helpers=True,
    include_anisotropy_toolbox=True,
    include_sc_pos=True,
)

from functions import download_data as download


In [ ]:
# ---------- User configuration ----------
cdf_lib_path = "/Applications/cdf/cdf/lib"  # update for your machine if needed
credentials = None

# Scan window (end is exclusive for daily slicing)
start_date = "2022-10-01 00:00"
end_date   = "2022-11-01 00:00"

# SOAR merged product options
SOLO_merged_fs = 256  # 256 or 4096
in_rtn = 1            # 1 -> RTN product, 0 -> SRF product

# Save options
save_usual_files = True
save_base = Path(root_dir) / "examples" / "downloaded_intervals" / "SOLO_merged_MAG_daily"
save_base.mkdir(parents=True, exist_ok=True)

settings = {
    "Data_path": Path(root_dir) / "data",
    "save_destination": Path(root_dir) / "examples" / "downloaded_intervals",
    "sc": "SOLO",
    "in_rtn": in_rtn,
    "use_local_data": False,

    # daily non-overlapping intervals
    "start_date": start_date,
    "end_date": end_date,
    "multiple_intervals": False,
    "duration": "24H",
    "Step": "24H",
    "addit_time_around": 1,

    # 1-minute products
    "part_resol": 60,
    "MAG_resol": 60,
    "upsample_low_freq_ts": False,

    "overwrite_files": 1,
    "must_have_qtn": False,
    "Max_par_missing": 30,
    "gap_time_threshold": 5,
    "save_all": True,

    "estimate_derived_param": True,
    "rol_window": "60min",

    # Required by your request
    "SOLO_use_merged_MAG": True,
    "SOLO_merged_fs": SOLO_merged_fs,

    "Big_Gaps": {
        "E_big_gaps": 10,
        "SC_pot_big_gaps": 10,
        "Mag_big_gaps": 500,
        "Par_big_gaps": 500,
        "QTN_big_gaps": 10,
    },

    "cut_in_small_windows": {
        "flag": False,
        "njobs": 1,
        "Step": "5s",
        "duration": "30s",
    },
}

vars_2_downnload = {"mag": None, "swa": None, "rpw": None, "ephem": None}

frame = "rtn" if in_rtn else "srf"
merged_product = f"multi-mag-rpw-scm-merged-{frame}-{SOLO_merged_fs}"
settings["SOLO_merged_product"] = merged_product

print("Save path:", save_base)
print("Merged product:", merged_product)


In [ ]:
# Daily non-overlapping intervals
start_ts = pd.Timestamp(start_date)
end_ts = pd.Timestamp(end_date)

edges = pd.date_range(start=start_ts, end=end_ts, freq="1D")
if len(edges) < 2:
    raise ValueError("Need at least 1 full day in [start_date, end_date].")

intervals = pd.DataFrame({"Start": edges[:-1], "End": edges[1:]})
print(f"Generated {len(intervals)} daily intervals.")
intervals.head()


## Phase 1 (fast): pre-screen with SOAR metadata only

This step checks archive availability by querying SOAR with `sunpy`/`sunpy_soar` search.
If `sunpy_soar` is unavailable in your environment, the notebook falls back to a conservative mode where all days are treated as candidates (still correct, just slower).


In [ ]:
def precheck_soar_availability(interval_df, product):
    rows = []
    try:
        from sunpy.net import Fido
        import sunpy.net.attrs as a
        import sunpy_soar  # noqa: F401
        soar_ok = True
    except Exception as e:
        print("SOAR metadata pre-check unavailable:", e)
        print("Falling back to full-run candidates for all days.")
        soar_ok = False

    if not soar_ok:
        for _, r in interval_df.iterrows():
            rows.append({
                "Start": r["Start"],
                "End": r["End"],
                "soar_available": 1,
                "soar_n_records": np.nan,
            })
        return pd.DataFrame(rows)

    for _, r in interval_df.iterrows():
        t0 = pd.Timestamp(r["Start"])
        t1 = pd.Timestamp(r["End"])

        try:
            qr = Fido.search(a.Time(t0, t1), a.soar.Product(product))
            nrec = len(qr)
            available = int(nrec > 0)
        except Exception:
            nrec = np.nan
            available = 0

        rows.append({
            "Start": t0,
            "End": t1,
            "soar_available": available,
            "soar_n_records": nrec,
        })

    return pd.DataFrame(rows)


precheck = precheck_soar_availability(intervals, merged_product)
print(
    f"SOAR candidate days: {int(precheck.soar_available.sum())}/{len(precheck)}"
)
precheck.head(20)


## Phase 2: run full diagnostics only on candidate days

For candidate days, run `download.main_function(...)`, then compute 1-minute summary statistics of `beta` and `sigma_c`.


In [ ]:
def _folder_name(start_time, end_time):
    tfmt = "%Y-%m-%d_%H-%M-%S"
    return f"{start_time.strftime(tfmt)}_{end_time.strftime(tfmt)}_sc_0"


def _safe_stat(series, reducer="mean"):
    if series is None:
        return np.nan
    series = pd.to_numeric(series, errors="coerce")
    if series.notna().sum() == 0:
        return np.nan
    if reducer == "mean":
        return float(np.nanmean(series.values))
    if reducer == "median":
        return float(np.nanmedian(series.values))
    return np.nan


rows = []
candidates = precheck[precheck["soar_available"] == 1].copy()

for _, row in candidates.iterrows():
    start_time = row["Start"]
    end_time = row["End"]

    (
        big_gaps_SC_pot,
        big_gaps,
        big_gaps_par,
        big_gaps_elec,
        big_gaps_qtn,
        flag_good,
        final_dict,
        general_dict,
        sig_df,
        dfdis,
        misc,
    ) = download.main_function(
        start_time,
        end_time,
        settings,
        vars_2_downnload,
        cdf_lib_path,
        credentials,
    )

    pipeline_ok = int(flag_good == 1 and isinstance(sig_df, pd.DataFrame) and len(sig_df) > 0)

    if pipeline_ok:
        sig_1min = sig_df.resample("1min").mean(numeric_only=True)
        beta = sig_1min.get("beta")
        sigma_c = sig_1min.get("sigma_c")

        beta_mean = _safe_stat(beta, "mean")
        beta_median = _safe_stat(beta, "median")
        sigma_c_abs_mean = _safe_stat(np.abs(sigma_c), "mean")
        sigma_c_abs_median = _safe_stat(np.abs(sigma_c), "median")

        if save_usual_files:
            folder = save_base / _folder_name(start_time, end_time)
            folder.mkdir(parents=True, exist_ok=True)

            pd.to_pickle(final_dict, folder / "final.pkl")
            pd.to_pickle(general_dict, folder / "general.pkl")
            pd.to_pickle(sig_df, folder / "sig_c_sig_r.pkl")
            pd.to_pickle(big_gaps, folder / "mag_gaps.pkl")
            pd.to_pickle(big_gaps_qtn, folder / "qtn_gaps.pkl")
            pd.to_pickle(big_gaps_par, folder / "par_gaps.pkl")
            pd.to_pickle(big_gaps_SC_pot, folder / "sc_pot_gaps.pkl")
            pd.to_pickle(big_gaps_elec, folder / "elec_gaps.pkl")
            pd.to_pickle(dfdis, folder / "distance.pkl")
            pd.to_pickle(misc, folder / "misc.pkl")
    else:
        beta_mean = np.nan
        beta_median = np.nan
        sigma_c_abs_mean = np.nan
        sigma_c_abs_median = np.nan

    rows.append({
        "Start": start_time,
        "End": end_time,
        "soar_available": int(row["soar_available"]),
        "soar_n_records": row["soar_n_records"],
        "pipeline_available": pipeline_ok,
        "beta_mean_1min": beta_mean,
        "beta_median_1min": beta_median,
        "abs_sigma_c_mean_1min": sigma_c_abs_mean,
        "abs_sigma_c_median_1min": sigma_c_abs_median,
    })

results_candidates = pd.DataFrame(rows)
results_candidates.head(20)


In [ ]:
# Merge phase-1 and phase-2 outputs into one final table
summary = precheck.merge(
    results_candidates,
    on=["Start", "End", "soar_available", "soar_n_records"],
    how="left",
)

summary["pipeline_available"] = summary["pipeline_available"].fillna(0).astype(int)

summary_path = save_base / "daily_merged_mag_availability_summary.csv"
summary.to_csv(summary_path, index=False)

print(f"SOAR-available days: {int(summary['soar_available'].sum())}/{len(summary)}")
print(f"Pipeline-success days: {int(summary['pipeline_available'].sum())}/{len(summary)}")
print(f"Saved summary: {summary_path}")

summary.head(30)


## Practical recommendation

This is the fastest robust pattern:

- Keep the SOAR pre-check as the default for long date ranges.
- Run full diagnostics only on SOAR-candidate days.
- If you only need a quick climatology, you can save only the summary CSV and skip writing per-day pickles.
